# Phase 3: Model Evaluation

This notebook evaluates the `best.pt` checkpoint of the Hybrid YOLO-Swin Weapon Detector. You can run this after training is completed to visualize detections and verify performance on the validation set.

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys

# Ensure the src/ and models/ directories are in the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

## 1. Load the Model
We load the `best.pt` checkpoint. If you just finished training, make sure the new `best.pt` is located at `../models/weights/best.pt` relative to this notebook.

In [ ]:
from models.hybrid_model import HybridWeaponDetector

WEIGHTS_PATH = "../models/weights/best.pt"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model on {device}...")
model = HybridWeaponDetector.load(WEIGHTS_PATH, device=device)
model.eval()
print("Model loaded successfully!")

## 2. Qualitative Evaluation (Visualization)
Let's run inference on a few sample images from the validation set to see how the model performs visually.

In [ ]:
def visualize_prediction(image_path, model, conf_threshold=0.25, iou_threshold=0.45):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not read {image_path}")
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Run inference
    detections = model.predict(img, conf_threshold=conf_threshold, iou_threshold=iou_threshold)
    
    # Draw boxes
    h, w = img.shape[:2]
    for det in detections:
        bbox = det['bbox']
        cls_name = det['class_name']
        conf = det['confidence']
        
        # The model's predict method handles normalized [0,1] vs pixel space based on its internal logic.
        # If coordinates are normalized (<= 1.0), scale them back.
        if all(x <= 1.0 for x in bbox) and w > 1:
            x1, y1, x2, y2 = [int(bbox[0]*w), int(bbox[1]*h), int(bbox[2]*w), int(bbox[3]*h)]
        else:
            x1, y1, x2, y2 = [int(x) for x in bbox]
        
        color = (255, 0, 0) if det.get('is_weapon') else (0, 255, 0)
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
        
        label = f"{cls_name} {conf:.2f}"
        cv2.putText(img_rgb, label, (x1, max(y1 - 10, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
    # Show
    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f"Predictions: {len(detections)} objects found")
    plt.show()

In [ ]:
import glob

# Point this to a directory containing some test/val images
VAL_IMAGES_DIR = "/content/yolo_dataset/yolo_dataset/val/images/"

if os.path.exists(VAL_IMAGES_DIR):
    sample_images = glob.glob(os.path.join(VAL_IMAGES_DIR, "*.jpg"))[:5]
    for img_path in sample_images:
        visualize_prediction(img_path, model, conf_threshold=0.25)
else:
    print(f"Validation directory not found at {VAL_IMAGES_DIR}. Please update the path or upload a sample image.")
    # Example with a specific image if you have one:
    # visualize_prediction('path_to_some_test_image.jpg', model)

## 3. Quantitative Evaluation (mAP)
To compute mAP across the entire validation set, we can iterate through the Ultralytics `YOLODataset` and compare the `HybridWeaponDetector` predictions against the ground truth labels.

> Note: Full mAP calculation over thousands of images takes time. This basic snippet demonstrates how to structure the evaluation loop.

In [ ]:
# Example logic for iterating the validation set
# Requires the ultralytics YOLODataset and metrics utilities if you want full mAP.
# 
# from ultralytics.data.dataset import YOLODataset
# from torch.utils.data import DataLoader
# 
# val_set = YOLODataset(img_path="/content/yolo_dataset/yolo_dataset/val/images", imgsz=640, augment=False, task='detect')
# val_loader = DataLoader(val_set, batch_size=8, shuffle=False)
# 
# for batch in val_loader:
#    images = batch['img'].to(device).float() / 255.0
#    preds = model(images)
#    decoded = model.head.decode_predictions(preds)
#    # ... compute IoU with batch['bboxes'] ...
